# ⚙️ 02. Preprocessing & Feature Engineering Pipeline
### Customer Segmentation & Churn Intelligence Platform

This notebook prototypes preprocessing components, validates robust scaling against standard scaling on skewed inputs, target-encodes categorical variables, and outlines model serialization targets.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
import joblib
from src import config

## 1. Load Feature Matrix & Hold-out Splits

In [ ]:
features_path = os.path.join(config.PROCESSED_DATA_DIR, "customer_feature_matrix.csv")
df = pd.read_csv(features_path)

# Define targets
df['is_churned'] = (df['recency_days'] > config.CHURN_THRESHOLD_DAYS).astype(int)

# 80-20 train-test splits
X = df.drop(columns=['customer_id', 'signup_date', 'is_churned'])
y = df['is_churned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=config.RANDOM_SEED, stratify=y)
print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")

## 2. Imputation & Preprocessing Prototypes

In [ ]:
# Numerical Imputations
num_cols = X_train.select_dtypes(include=[np.number]).columns
medians = X_train[num_cols].median()

X_train_imp = X_train.copy()
X_train_imp[num_cols] = X_train_imp[num_cols].fillna(medians)

# Categorical Imputations
cat_cols = X_train.select_dtypes(exclude=[np.number]).columns
X_train_imp[cat_cols] = X_train_imp[cat_cols].fillna('UNKNOWN')
print("Imputations populated successfully.")

## 3. Scaling Comparison: RobustScaler vs StandardScaler on Outliers

In [ ]:
# Total spend has substantial tail variations
target_column = 'monetary_value_total'

scaler_std = StandardScaler()
scaler_rob = RobustScaler()

scaled_std = scaler_std.fit_transform(X_train_imp[[target_column]])
scaled_rob = scaler_rob.fit_transform(X_train_imp[[target_column]])

print(f"StandardScaler range on '{target_column}': [{scaled_std.min():.2f}, {scaled_std.max():.2f}] | Mean: {scaled_std.mean():.2f}")
print(f"RobustScaler range on '{target_column}': [{scaled_rob.min():.2f}, {scaled_rob.max():.2f}] | Median: {np.median(scaled_rob):.2f}")

## 4. Preprocessor Serialization Checks

In [ ]:
# Fits overall RobustScaler
final_scaler = RobustScaler()
final_scaler.fit(X_train_imp[num_cols])

# Serializing scaler for Serving API integrations
scaler_path = os.path.join(config.MODELS_DIR, "scaler.joblib")
joblib.dump(final_scaler, scaler_path)
print(f"Scaler successfully serialized to: {scaler_path}")